# Module 07 — Notebook 4: Mini-Project — Reproducible Analysis Setup

## What You'll Build

By the end of this notebook, you'll have created:

- `scripts/analyze_scores.py` — an analysis script with seeds set at the top
- `requirements.txt` — exact pinned package versions for the environment
- `REPRODUCE.md` — step-by-step instructions for anyone to reproduce your results

You'll verify reproducibility by running the script twice and confirming the outputs are identical.

**Estimated time:** ~25 minutes

## Why This Matters for AI Research Engineering

These three artifacts — a seeded script, a pinned requirements file, and a reproduction README — are the minimum for a reproducible experiment. Real AI safety papers ship repositories with exactly this structure. NeurIPS reproducibility checklists and Anthropic's model cards both require this level of documentation.

The REPRODUCE.md is what lets a colleague (or a reviewer) run your experiment without emailing you for help.

In [ ]:
import sys
sys.path.insert(0, "../../")
from src.checks import check_equal, check_type, check_contains
import subprocess
import json
import random
import numpy as np
import importlib.metadata
from pathlib import Path

# Create directories needed for this project
SCRIPTS_DIR = Path("scripts")
OUTPUT_DIR = Path("output")
SCRIPTS_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
print("Directories ready. Let's build something reproducible.")

## Step 1: The Analysis Script

First, let's look at the helper functions you'll use inside the script. These load model outputs, compute summary statistics, and take a reproducible random sample.

In [ ]:
# These functions belong inside your script — study them before writing it

def load_outputs(path):
    """Load a JSON file of model outputs."""
    with open(path) as f:
        return json.load(f)

def compute_stats(outputs):
    """Compute mean, min, max of scores across all outputs."""
    scores = [o["score"] for o in outputs]
    return {
        "n": len(scores),
        "mean": round(sum(scores) / len(scores), 4),
        "min": round(min(scores), 4),
        "max": round(max(scores), 4),
    }

def sample_outputs(outputs, n=5, seed=42):
    """Randomly sample n outputs. Uses a local Random instance so it doesn't interfere with global state."""
    rng = random.Random(seed)
    return rng.sample(outputs, min(n, len(outputs)))

# Quick test with synthetic data
dummy = [{"score": i * 0.1} for i in range(10)]
print("Stats:", compute_stats(dummy))
print("Sample IDs:", [dummy.index(o) for o in sample_outputs(dummy, n=3)])

Now write the full script. It must:

1. Import `random`, `numpy as np`, `argparse`, `json`, `logging`, and `Path` from `pathlib`
2. Define `SEED = 42` at module level
3. Call `random.seed(SEED)` and `np.random.seed(SEED)` before any stochastic code
4. Define the helper functions above (`load_outputs`, `compute_stats`, `sample_outputs`)
5. Define `main()` that:
   - Parses `--input` (required, `type=Path`) and `--output` (default `Path("output/stats.json")`)
   - Loads the JSON file with `load_outputs()`
   - Computes stats with `compute_stats()`
   - Samples 5 outputs with `sample_outputs(outputs, n=5, seed=SEED)`
   - Saves `{"stats": ..., "sample": [...]}` to `--output` as JSON
   - Logs: `"Loaded N outputs"` and `"Stats: ..."`
6. Has the `if __name__ == "__main__":` guard

In [ ]:
%%writefile scripts/analyze_scores.py
"""
analyze_scores.py — Reproducible model output analysis.

Usage:
    python scripts/analyze_scores.py --input ../../data/synthetic/model_outputs.json
"""
# YOUR CODE HERE
# Replace this stub with the full script as described above.
print("replace me")

In [ ]:
script_path = Path("scripts/analyze_scores.py")
check_equal(script_path.exists(), True, "scripts/analyze_scores.py exists")

source = script_path.read_text()
check_contains(source, "SEED", "SEED constant is defined")
check_contains(source, "random.seed", "random.seed is called")
check_contains(source, "np.random.seed", "np.random.seed is called")
check_contains(source, "def main", "main() function is defined")
check_contains(source, "__name__", "entry-point guard exists")
check_contains(source, "argparse", "argparse is used")
check_contains(source, "--input", "--input argument is defined")

## Step 2: Verify Reproducibility — Run the Script Twice

The acid test: run the script twice and confirm the output files are identical. If they differ, something isn't seeded.

In [ ]:
DATA = Path("../../data/synthetic/model_outputs.json")
OUTPUT_1 = Path("output/run1_stats.json")
OUTPUT_2 = Path("output/run2_stats.json")

result1 = subprocess.run(
    [sys.executable, "scripts/analyze_scores.py", "--input", str(DATA), "--output", str(OUTPUT_1)],
    capture_output=True, text=True
)
result2 = subprocess.run(
    [sys.executable, "scripts/analyze_scores.py", "--input", str(DATA), "--output", str(OUTPUT_2)],
    capture_output=True, text=True
)

print("Run 1 exit code:", result1.returncode)
if result1.stderr:
    print("Run 1 stderr:", result1.stderr[:500])

print("Run 2 exit code:", result2.returncode)
if result2.stderr:
    print("Run 2 stderr:", result2.stderr[:500])

In [ ]:
# YOUR CODE HERE
# Load both output files and compare them.
# Store True in runs_are_identical if the outputs are equal, False otherwise.

runs_are_identical = None  # bool

In [ ]:
check_equal(result1.returncode, 0, "run 1 exited successfully")
check_equal(result2.returncode, 0, "run 2 exited successfully")
check_equal(runs_are_identical, True, "both runs produce identical output (reproducibility verified!)")

## Step 3: Pin Your Dependencies

Use `importlib.metadata` to find the exact installed versions, then write a `requirements.txt`.

In [ ]:
# Find installed versions to use in your requirements.txt
import importlib.metadata

for pkg in ["numpy", "pandas", "matplotlib"]:
    try:
        v = importlib.metadata.version(pkg)
        print(f"{pkg}=={v}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{pkg} — not installed")

In [ ]:
%%writefile requirements.txt
# YOUR REQUIREMENTS HERE
# Use the exact versions printed above (== for all packages)
# Include at least numpy and pandas


In [ ]:
req_path = Path("requirements.txt")
check_equal(req_path.exists(), True, "requirements.txt exists")

content = req_path.read_text()
check_contains(content, "numpy==", "numpy is pinned with ==")
check_contains(content, "pandas==", "pandas is pinned with ==")

pinned_lines = [l for l in content.splitlines() if "==" in l]
check_equal(len(pinned_lines) >= 2, True, "at least 2 packages are pinned")

## Step 4: Write the Reproduction Guide

A `REPRODUCE.md` tells someone unfamiliar with your work exactly how to go from zero to running your analysis. It must include:

1. A `## Setup` section with commands to create and activate a venv
2. A `## Install Dependencies` section with the install command
3. A `## Run Analysis` section with the exact command to run `analyze_scores.py`
4. A note about what output to expect

In [ ]:
%%writefile REPRODUCE.md
# Reproduction Guide

<!-- YOUR CONTENT HERE -->
<!-- Replace this comment with the three required sections -->

In [ ]:
md_path = Path("REPRODUCE.md")
check_equal(md_path.exists(), True, "REPRODUCE.md exists")

content = md_path.read_text()
check_contains(content, "## Setup", "REPRODUCE.md has a ## Setup section")
check_contains(content, "requirements.txt", "REPRODUCE.md references requirements.txt")
check_contains(content, "analyze_scores.py", "REPRODUCE.md references the analysis script")

## Wrap-Up

You've built the standard reproducible experiment scaffold:

| Artifact | Purpose |
|---|---|
| `scripts/analyze_scores.py` | The analysis with `SEED` and both seeds set at the top |
| `requirements.txt` | Exact package versions — the environment contract |
| `REPRODUCE.md` | Step-by-step instructions for anyone starting from scratch |
| Two identical runs | Proof that the setup is actually deterministic |

This is the minimum reproducibility bar for AI research. Real experiments may also require:
- Pinning the Python version itself (use `.python-version` with `pyenv`)
- Recording hardware and OS (for GPU-dependent results)
- Archiving the dataset used

**Next module:** Module 08 — Basic Statistics for Experiments